## 연습 1: Solaris Energy 데이터 플랫폼 설정하기

데이터를 수집하려면 먼저 Unity Catalog 구조를 구성해야 합니다. 플랫폼을 나타내는 **카탈로그**, 처리 계층별로 데이터를 정리하는 **스키마**, 원시 파일을 저장하는 **볼륨**이 필요합니다.

이 연습을 완료한 후에는 랩 설정 페이지로 돌아가 **Catalog Explorer**에서 만든 객체를 확인하세요.

### 작업 1: Unity Catalog 구조 설정하기

이전 랩에서 이미 카탈로그, 스키마, 볼륨 만들기를 연습했으므로, 이 설정 코드는 제공됩니다 — 아래 셀을 실행하기만 하면 환경이 준비됩니다.

이 코드는 다음 Unity Catalog 객체를 만듭니다:
- `solaris_lab`이라는 이름의 **카탈로그** — Solaris Energy 플랫폼의 최상위 네임스페이스
- `solaris_lab` 내부의 **bronze 스키마** — 원본 시스템에서 도착한 그대로의 원시 수집 데이터용
- `solaris_lab` 내부의 **silver 스키마** — 정제 및 보강된 데이터용
- `solaris_lab.bronze` 내부의 `raw_files`라는 이름의 **관리형 볼륨** — 현장 센서 및 SCADA 시스템에서 들어오는 CSV 파일의 랜딩 존(landing zone)

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS solaris_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Solaris Energy 분석 플랫폼 — 유럽 전역의 태양광 발전소 및 풍력 터빈'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS solaris_lab.bronze
  COMMENT '원시 수집 데이터 — 현장 시스템에서 받은 그대로, 수정되지 않음';

CREATE SCHEMA IF NOT EXISTS solaris_lab.silver
  COMMENT '정제 및 보강된 데이터 — 분석 및 보고 준비 완료';
  
CREATE VOLUME IF NOT EXISTS solaris_lab.bronze.raw_files
  COMMENT 'SCADA 시스템 및 현장 센서의 원시 CSV 파일을 위한 랜딩 존';

### ✅ 제공됨: 샘플 데이터로 볼륨 채우기

아래 셀을 실행하여 두 개의 샘플 CSV 파일을 볼륨에 기록합니다:
- `solar_readings.csv` — 스페인과 독일 사이트의 태양광 패널 출력 측정값
- `turbine_events.csv` — 네덜란드와 덴마크의 풍력 터빈 유지보수 및 경고 이벤트

이 파일들은 연습 2, 3, 4에서 사용합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 샘플 데이터 파일을 볼륨에 기록합니다

solar_csv = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R001,SITE_ES_001,P-ES-001,2025-06-01 06:30:00,8.2,210,18.4,Normal
R002,SITE_ES_001,P-ES-002,2025-06-01 06:30:00,7.9,205,18.6,Normal
R003,SITE_ES_001,P-ES-003,2025-06-01 06:30:00,8.5,215,18.2,Normal
R004,SITE_ES_001,P-ES-004,2025-06-01 06:30:00,0.0,205,18.5,Fault
R005,SITE_ES_001,P-ES-005,2025-06-01 06:30:00,7.6,200,18.8,Normal
R006,SITE_ES_001,P-ES-001,2025-06-01 09:00:00,142.3,750,28.1,Normal
R007,SITE_ES_001,P-ES-002,2025-06-01 09:00:00,138.7,745,28.4,Normal
R008,SITE_ES_001,P-ES-003,2025-06-01 09:00:00,145.1,755,27.9,Normal
R009,SITE_ES_001,P-ES-004,2025-06-01 09:00:00,0.0,750,28.2,Fault
R010,SITE_ES_001,P-ES-005,2025-06-01 09:00:00,136.4,740,28.7,Normal
R011,SITE_ES_001,P-ES-001,2025-06-01 12:00:00,228.5,1150,41.3,Normal
R012,SITE_ES_001,P-ES-002,2025-06-01 12:00:00,221.4,1140,41.8,Warning
R013,SITE_ES_001,P-ES-003,2025-06-01 12:00:00,235.8,1160,40.9,Normal
R014,SITE_ES_001,P-ES-004,2025-06-01 12:00:00,0.0,1150,41.5,Fault
R015,SITE_ES_001,P-ES-005,2025-06-01 12:00:00,219.6,1135,42.1,Warning
R016,SITE_DE_001,P-DE-001,2025-06-01 06:30:00,4.1,120,12.3,Normal
R017,SITE_DE_001,P-DE-002,2025-06-01 06:30:00,4.3,125,12.1,Normal
R018,SITE_DE_001,P-DE-003,2025-06-01 06:30:00,3.8,118,12.5,Normal
R019,SITE_DE_001,P-DE-004,2025-06-01 06:30:00,4.0,122,12.4,Normal
R020,SITE_DE_001,P-DE-005,2025-06-01 06:30:00,4.2,124,12.2,Normal
R021,SITE_DE_001,P-DE-001,2025-06-01 09:00:00,98.7,540,19.8,Normal
R022,SITE_DE_001,P-DE-002,2025-06-01 09:00:00,101.2,548,19.6,Normal
R023,SITE_DE_001,P-DE-003,2025-06-01 09:00:00,96.4,535,20.1,Normal
R024,SITE_DE_001,P-DE-004,2025-06-01 09:00:00,99.8,542,19.9,Normal
R025,SITE_DE_001,P-DE-005,2025-06-01 09:00:00,102.1,550,19.4,Normal
R026,SITE_DE_001,P-DE-001,2025-06-01 12:00:00,187.3,980,31.2,Normal
R027,SITE_DE_001,P-DE-002,2025-06-01 12:00:00,191.6,992,30.8,Normal
R028,SITE_DE_001,P-DE-003,2025-06-01 12:00:00,184.9,975,31.5,Normal
R029,SITE_DE_001,P-DE-004,2025-06-01 12:00:00,188.7,984,31.0,Normal
R030,SITE_DE_001,P-DE-005,2025-06-01 12:00:00,193.2,996,30.6,Normal
R031,SITE_ES_001,P-ES-001,2025-06-02 12:00:00,232.1,1165,42.7,Normal
R032,SITE_ES_001,P-ES-002,2025-06-02 12:00:00,226.9,1155,43.1,Normal
R033,SITE_ES_001,P-ES-003,2025-06-02 12:00:00,239.4,1175,42.3,Normal
R034,SITE_ES_001,P-ES-004,2025-06-02 12:00:00,225.2,1148,43.4,Normal
R035,SITE_ES_001,P-ES-005,2025-06-02 12:00:00,231.7,1162,42.9,Normal
"""

turbine_csv = """event_id,turbine_id,site_id,event_timestamp,event_type,severity,description,resolved
EVT001,T-NL-001,WIND_NL_001,2025-06-01 02:14:00,Alert,Medium,Vibration level 18% above baseline threshold,true
EVT002,T-NL-002,WIND_NL_001,2025-06-01 03:47:00,Fault,High,Gearbox oil temperature exceeded 85 degrees C,true
EVT003,T-NL-003,WIND_NL_001,2025-06-01 07:30:00,Inspection,Low,Scheduled 6-month rotor blade inspection,true
EVT004,T-NL-004,WIND_NL_001,2025-06-01 09:12:00,Alert,Low,Wind speed sensor calibration drift detected,false
EVT005,T-NL-005,WIND_NL_001,2025-06-01 11:55:00,Maintenance,Medium,Yaw motor lubrication service,true
EVT006,T-DK-001,WIND_DK_001,2025-06-01 01:05:00,Alert,Critical,Blade pitch control failure - emergency stop activated,true
EVT007,T-DK-002,WIND_DK_001,2025-06-01 04:22:00,Fault,Medium,Power converter overload - output capped at 60%,false
EVT008,T-DK-003,WIND_DK_001,2025-06-01 06:45:00,Inspection,Low,Annual safety and compliance inspection,true
EVT009,T-DK-004,WIND_DK_001,2025-06-01 10:33:00,Maintenance,Low,Tower bolt torque check completed,true
EVT010,T-DK-005,WIND_DK_001,2025-06-01 14:18:00,Alert,Medium,Ice accretion on blades - de-icing initiated,true
EVT011,T-NL-001,WIND_NL_001,2025-06-02 05:30:00,Maintenance,Low,Hydraulic fluid top-up completed,true
EVT012,T-NL-002,WIND_NL_001,2025-06-02 08:00:00,Alert,High,Generator winding temperature 12 degrees C above limit,false
EVT013,T-NL-003,WIND_NL_001,2025-06-02 09:45:00,Fault,Medium,Anemometer replaced after sensor malfunction,true
EVT014,T-NL-004,WIND_NL_001,2025-06-02 12:10:00,Inspection,Low,Control system firmware update applied,true
EVT015,T-NL-005,WIND_NL_001,2025-06-02 15:22:00,Alert,Low,Brake pad wear indicator triggered,false
EVT016,T-DK-001,WIND_DK_001,2025-06-02 03:18:00,Maintenance,Medium,Main bearing grease replenishment,true
EVT017,T-DK-002,WIND_DK_001,2025-06-02 07:55:00,Alert,High,Pitch battery backup low - replacement scheduled,false
EVT018,T-DK-003,WIND_DK_001,2025-06-02 10:40:00,Fault,Critical,Transformer fault - turbine offline for 4 hours,true
EVT019,T-DK-004,WIND_DK_001,2025-06-02 13:15:00,Inspection,Low,Cooling system filters cleaned,true
EVT020,T-DK-005,WIND_DK_001,2025-06-02 16:50:00,Maintenance,Medium,Slip ring inspection and cleaning completed,true
"""

with open("/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv", "w") as f:
    f.write(solar_csv.strip())

with open("/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv", "w") as f:
    f.write(turbine_csv.strip())

print("✅ 샘플 데이터를 /Volumes/solaris_lab/bronze/raw_files/에 기록했습니다")

## 연습 2: PySpark DataFrame을 사용한 배치 수집

Solaris Energy의 SCADA 시스템은 매일 태양광 패널 측정값을 CSV 파일로 내보냅니다. 첫 번째 수집 작업은 볼륨에서 최신 파일을 읽고, 그 내용을 검사한 다음, `bronze` 계층에 관리형 Delta 테이블로 기록하는 것입니다.

이 연습을 완료한 후에는 랩 설정 페이지로 돌아가 **Catalog Explorer**에서 테이블을 살펴보세요.

### 작업 2.1 — 태양광 측정값 CSV를 DataFrame으로 읽기

`spark.read` API를 사용하여 볼륨에서 `solar_readings.csv`를 적재합니다. 파일에는 헤더 행이 있으며, Spark가 컬럼 데이터 타입을 자동으로 추론하도록 해야 합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"헤더와 스키마 추론을 활성화하여 Unity Catalog 볼륨의 CSV 파일을 PySpark DataFrame으로 어떻게 읽나요?"*

**힌트:**
- 볼륨 경로는 `/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv`입니다
- `.format("csv")`, `.option("header", "true")`, `.option("inferSchema", "true")`, `.load(...)`를 사용하세요

In [ ]:
df_solar = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv")
)

### 작업 2.2 — 스키마 및 샘플 행 검사하기

데이터를 테이블에 기록하기 전에 추론된 스키마를 검증하고 레코드 샘플을 확인하는 것이 좋습니다. `power_kw`, `irradiance_w_m2`, `temperature_c`가 숫자 타입인지 확인하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"PySpark DataFrame의 스키마를 출력하고 처음 몇 개의 행을 표시하려면 어떻게 하나요?"*

**힌트:**
- 컬럼 이름과 데이터 타입을 보려면 `df_solar.printSchema()`를 사용하세요
- 데이터를 미리 보려면 `display(df_solar)`를 사용하세요

In [ ]:
# 스키마 출력 — power_kw, irradiance_w_m2, temperature_c가 숫자 타입인지 확인
df_solar.printSchema()

# 데이터 샘플 표시
display(df_solar)

### 작업 2.3 — DataFrame을 Delta 테이블에 기록하기

태양광 측정값을 `solaris_lab.bronze.solar_readings`에 관리형 Delta 테이블로 저장합니다. Azure Databricks는 기본적으로 Delta Lake 형식을 사용하며, 이는 ACID 트랜잭션과 time travel을 제공합니다.

셀을 실행할 때마다 테이블이 완전히 교체되도록 `overwrite` 모드를 사용하세요(초기 전체 적재에 적합).

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"saveAsTable을 overwrite 모드로 사용하여 PySpark DataFrame을 Unity Catalog Delta 테이블에 어떻게 기록하나요?"*

**힌트:**
- `df_solar.write.mode("overwrite").saveAsTable("solaris_lab.bronze.solar_readings")`를 사용하세요

In [ ]:
df_solar.write.mode("overwrite").saveAsTable("solaris_lab.bronze.solar_readings")

print("✅ 태양광 측정값을 solaris_lab.bronze.solar_readings에 기록했습니다")

### 작업 2.4 — 수집된 테이블 검증하기

방금 만든 Delta 테이블을 조회하여 모든 행이 적재되었는지 확인합니다. 또한 `status = 'Fault'`인 행을 필터링하여 패널 `P-ES-004`의 오류(fault) 측정값이 존재하는지 확인하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks SQL 테이블에서 행 수를 세고 컬럼 값으로 필터링하려면 어떻게 하나요?"*

**힌트:**
- `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings`를 사용하세요
- 오류를 필터링하려면 `WHERE status = 'Fault'` 절을 추가하세요

In [ ]:
%sql
-- 전체 행 수 — 35여야 합니다
SELECT COUNT(*) AS total_readings FROM solaris_lab.bronze.solar_readings;

In [ ]:
%sql
-- 오류(Fault) 측정값 — 3건이어야 합니다 (P-ES-004의 R004, R009, R014)
SELECT reading_id, panel_id, reading_timestamp, power_kw, status
FROM solaris_lab.bronze.solar_readings
WHERE status = 'Fault'
ORDER BY reading_timestamp;

## 연습 3: SQL 기반 수집

Solaris Energy의 운영팀은 시간이 지남에 따라 볼륨에 누적되는 CSV 파일로 풍력 터빈 이벤트 로그를 전달합니다. 실행할 때마다 전체 파일을 다시 적재하는 대신, 멱등적(idempotent)이고 증분적인 적재를 위해 `COPY INTO`를 사용합니다. 그런 다음 `CREATE TABLE AS SELECT`를 사용하여 silver 계층 요약 테이블을 만듭니다.

### 작업 3.1 — 터빈 이벤트 대상 테이블 만들기

`COPY INTO`를 실행하려면 대상 테이블이 먼저 존재해야 합니다. 다음 컬럼으로 Delta 테이블 `solaris_lab.bronze.turbine_events`를 만듭니다:

| Column | Type |
|---|---|
| `event_id` | `STRING` |
| `turbine_id` | `STRING` |
| `site_id` | `STRING` |
| `event_timestamp` | `TIMESTAMP` |
| `event_type` | `STRING` |
| `severity` | `STRING` |
| `description` | `STRING` |
| `resolved` | `BOOLEAN` |

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"CREATE TABLE IF NOT EXISTS를 사용하여 명시적 컬럼 타입으로 Databricks SQL에서 Delta 테이블을 어떻게 만드나요?"*

**힌트:**
- `CREATE TABLE IF NOT EXISTS solaris_lab.bronze.turbine_events (...)`를 사용하세요
- `USING DELTA`를 사용하세요 (Azure Databricks에서 권장되는 스토리지 형식)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS solaris_lab.bronze.turbine_events (
  event_id        STRING,
  turbine_id      STRING,
  site_id         STRING,
  event_timestamp TIMESTAMP,
  event_type      STRING,
  severity        STRING,
  description     STRING,
  resolved        BOOLEAN
)
USING DELTA
COMMENT 'Solaris Energy 사이트의 풍력 터빈 유지보수 및 경고 이벤트';

### 작업 3.2 — COPY INTO로 터빈 이벤트 적재하기

`COPY INTO`를 사용하여 볼륨에서 `turbine_events.csv`를 적재합니다. `COPY INTO`는 멱등적입니다 — 이미 적재된 파일을 추적하므로, 명령을 다시 실행해도 중복 행이 생성되지 않습니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"헤더와 타입 추론을 위한 FORMAT_OPTIONS를 포함하여, Databricks SQL에서 COPY INTO를 사용하여 Unity Catalog 볼륨의 CSV 파일을 어떻게 적재하나요?"*

**힌트:**
- `FROM` 절을 `/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv`로 지정하세요
- `FILEFORMAT = CSV`로 설정하세요
- `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')`를 사용하세요

In [ ]:
%sql
COPY INTO solaris_lab.bronze.turbine_events
FROM '/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

### 작업 3.3 — 멱등성 검증하기

지금 `turbine_events`의 행 수를 세고, `COPY INTO`를 다시 실행한 다음, 행 수를 한 번 더 세어 보세요. 두 번 모두 동일한 개수가 표시되어야 합니다 — 이는 `COPY INTO`가 이미 적재된 파일을 건너뛴다는 것을 보여줍니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 COPY INTO는 이미 적재된 파일을 어떻게 추적하나요?"*

**힌트:**
- `COPY INTO`를 다시 실행하기 전과 후에 `SELECT COUNT(*) FROM solaris_lab.bronze.turbine_events`를 사용하세요
- 두 개수 모두 동일한 숫자를 반환해야 합니다

In [ ]:
%sql
-- 첫 번째 개수 — 20이어야 합니다
SELECT COUNT(*) AS event_count FROM solaris_lab.bronze.turbine_events;

In [ ]:
%sql
-- COPY INTO 다시 실행 (작업 3.2와 동일한 명령)
COPY INTO solaris_lab.bronze.turbine_events
FROM '/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

In [ ]:
%sql
-- 두 번째 개수 — 여전히 20이어야 합니다 (중복 없음)
SELECT COUNT(*) AS event_count FROM solaris_lab.bronze.turbine_events;

### 작업 3.4 — CTAS로 silver 요약 테이블 만들기

에너지 분석팀은 각 사이트의 출력을 추적하기 위한 일일 생산 요약이 필요합니다. `CREATE TABLE AS SELECT`(`CTAS`)를 사용하여 bronze `solar_readings` 테이블에서 집계하여 `solaris_lab.silver.daily_solar_production`을 만듭니다.

요약에는 다음이 포함되어야 합니다:
- `reading_date` — `reading_timestamp`의 날짜 부분
- `site_id`
- `total_power_kw` — Normal 상태 측정값만의 `power_kw` 합계
- `avg_irradiance` — `irradiance_w_m2`의 평균
- `fault_count` — `status = 'Fault'`인 측정값의 개수

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks SQL에서 CREATE TABLE AS SELECT를 사용하여 기존 테이블에서 데이터를 집계하고 날짜와 사이트별로 그룹화하려면 어떻게 하나요?"*

**힌트:**
- `CREATE OR REPLACE TABLE solaris_lab.silver.daily_solar_production AS SELECT ...`를 사용하세요
- 날짜를 추출하려면 `CAST(reading_timestamp AS DATE)` 또는 `DATE(reading_timestamp)`를 사용하세요
- 조건부 집계에는 `SUM(CASE WHEN status = 'Normal' THEN power_kw ELSE 0 END)`를 사용하세요
- `reading_date`와 `site_id`로 그룹화하세요

In [ ]:
%sql
CREATE OR REPLACE TABLE solaris_lab.silver.daily_solar_production AS
SELECT
  DATE(reading_timestamp)                                              AS reading_date,
  site_id,
  ROUND(SUM(CASE WHEN status = 'Normal' THEN power_kw ELSE 0 END), 2) AS total_power_kw,
  ROUND(AVG(irradiance_w_m2), 1)                                      AS avg_irradiance,
  COUNT(CASE WHEN status = 'Fault' THEN 1 END)                        AS fault_count
FROM solaris_lab.bronze.solar_readings
GROUP BY reading_date, site_id
ORDER BY reading_date, site_id;

In [ ]:
%sql
SELECT * FROM solaris_lab.silver.daily_solar_production
ORDER BY reading_date, site_id;

## 연습 4: Auto Loader를 사용한 연속 파일 수집

새로운 태양광 측정값은 SCADA 시스템에서 15분마다 도착하여 클라우드 스토리지 디렉터리에 개별 CSV 파일로 저장됩니다. 새 파일을 스캔하기 위해 배치 작업을 예약하는 대신, **Auto Loader**를 사용하여 파일을 자동으로 감지하고 수집합니다 — 각 파일을 정확히 한 번씩 처리합니다.

Auto Loader는 `spark.readStream`과 함께 `cloudFiles` 소스 형식을 사용합니다. `trigger(availableNow=True)` 옵션은 현재 사용 가능한 모든 파일을 처리한 후 중지하므로, 예약된 수집이나 온디맨드 수집에 이상적입니다.

### ✅ 제공됨: 들어오는 데이터 파일 시뮬레이션

아래 셀을 실행하세요. 볼륨의 `incoming/` 하위 디렉터리에 세 개의 개별 CSV 파일을 만들어, 연속된 세 번의 SCADA 내보내기 웨이브를 시뮬레이션합니다. Auto Loader가 이들을 모두 감지하고 처리합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 들어오는 데이터 파일 세 개를 기록합니다

import os

incoming_dir = "/Volumes/solaris_lab/bronze/raw_files/incoming"
os.makedirs(incoming_dir, exist_ok=True)

wave1 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R036,SITE_DE_001,P-DE-001,2025-06-03 09:00:00,103.4,558,20.3,Normal
R037,SITE_DE_001,P-DE-002,2025-06-03 09:00:00,99.7,545,20.6,Normal
R038,SITE_DE_001,P-DE-003,2025-06-03 09:00:00,105.1,562,20.1,Normal
R039,SITE_DE_001,P-DE-004,2025-06-03 09:00:00,101.3,552,20.4,Warning
R040,SITE_DE_001,P-DE-005,2025-06-03 09:00:00,97.8,540,20.8,Normal"""

wave2 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R041,SITE_ES_001,P-ES-001,2025-06-03 09:15:00,148.2,765,29.4,Normal
R042,SITE_ES_001,P-ES-002,2025-06-03 09:15:00,143.5,755,29.8,Normal
R043,SITE_ES_001,P-ES-003,2025-06-03 09:15:00,151.9,778,29.1,Normal
R044,SITE_ES_001,P-ES-004,2025-06-03 09:15:00,146.7,761,29.5,Normal
R045,SITE_ES_001,P-ES-005,2025-06-03 09:15:00,140.3,748,30.1,Normal"""

wave3 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R046,SITE_DE_001,P-DE-001,2025-06-03 09:30:00,108.6,570,20.9,Normal
R047,SITE_DE_001,P-DE-002,2025-06-03 09:30:00,104.9,562,21.2,Normal
R048,SITE_DE_001,P-DE-003,2025-06-03 09:30:00,110.3,575,20.7,Normal
R049,SITE_DE_001,P-DE-004,2025-06-03 09:30:00,0.0,565,21.0,Fault
R050,SITE_DE_001,P-DE-005,2025-06-03 09:30:00,102.7,558,21.4,Normal"""

for name, content in [("wave_1.csv", wave1), ("wave_2.csv", wave2), ("wave_3.csv", wave3)]:
    with open(f"{incoming_dir}/{name}", "w") as f:
        f.write(content)

print(f"✅ 들어오는 파일 3개를 {incoming_dir}/에 기록했습니다")
print("   wave_1.csv — SITE_DE_001의 09:00 측정값 5건")
print("   wave_2.csv — SITE_ES_001의 09:15 측정값 5건")
print("   wave_3.csv — SITE_DE_001의 09:30 측정값 5건")

### 작업 4.1 — Auto Loader 읽기 스트림 구성하기

`spark.readStream`과 `cloudFiles` 형식을 사용하여 `incoming/` 디렉터리에서 새 CSV 파일을 모니터링하는 스트리밍 DataFrame을 만듭니다. Auto Loader는 이미 처리한 파일을 자동으로 추적합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"schemaLocation과 함께 spark.readStream 및 cloudFiles 형식을 사용하여 Unity Catalog 볼륨의 CSV 파일을 읽도록 PySpark에서 Auto Loader를 어떻게 구성하나요?"*

**힌트:**
- `spark.readStream`에 `.format("cloudFiles")`를 사용하세요
- `cloudFiles.format`을 `csv`로 설정하세요
- `cloudFiles.schemaLocation`을 볼륨 내부 경로로 설정하세요(예: `/Volumes/solaris_lab/bronze/raw_files/schema`)
- 숫자 컬럼이 올바르게 타입 지정되도록 `cloudFiles.inferColumnTypes`를 `true`로 설정하세요
- `/Volumes/solaris_lab/bronze/raw_files/incoming/`에서 적재하세요

In [ ]:
df_autoloader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/solaris_lab/bronze/raw_files/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load("/Volumes/solaris_lab/bronze/raw_files/incoming/")
)

### 작업 4.2 — 스트림을 Delta 테이블에 기록하기

Auto Loader 스트림을 `solaris_lab.bronze.solar_readings_stream`에 기록합니다. 현재 사용 가능한 모든 파일을 처리한 후 중지하도록 `trigger(availableNow=True)`를 사용하세요 — 이는 예약된 수집 작업에 적합한 패턴입니다.

또한 작업이 다시 시작될 때 Auto Loader가 진행 상황을 추적하고 올바르게 재개할 수 있도록 **체크포인트 위치(checkpoint location)**를 지정해야 합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"writeStream, trigger(availableNow=True), 체크포인트 위치를 사용하여 Auto Loader(spark.readStream cloudFiles) 스트림을 Unity Catalog 테이블에 어떻게 기록하나요?"*

**힌트:**
- 스트리밍 DataFrame에 `.writeStream`을 사용하세요
- `checkpointLocation`을 `/Volumes/solaris_lab/bronze/raw_files/checkpoint`로 설정하세요
- 대기 중인 모든 파일을 처리한 후 중지하려면 `.trigger(availableNow=True)`를 사용하세요
- 싱크로 `.toTable("solaris_lab.bronze.solar_readings_stream")`을 사용하세요
- 완료를 기다리려면 쿼리 객체에서 `.awaitTermination()`을 호출하세요

In [ ]:
query = (
    df_autoloader.writeStream
    .option("checkpointLocation", "/Volumes/solaris_lab/bronze/raw_files/checkpoint")
    .trigger(availableNow=True)
    .toTable("solaris_lab.bronze.solar_readings_stream")
)

query.awaitTermination()
print("✅ Auto Loader 스트림이 완료되었습니다")

### 작업 4.3 — Auto Loader 수집 검증하기

`solaris_lab.bronze.solar_readings_stream`을 조회하여 15개 행(웨이브당 5개 × 3개 웨이브)이 모두 적재되었는지 확인합니다. 그런 다음 write stream 셀을 한 번 더 실행하여 중복 행이 생성되지 않는지 확인합니다 — 이는 Auto Loader의 정확히 한 번(exactly-once) 보장을 보여줍니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Auto Loader가 올바른 수의 행을 수집했고 중복을 생성하지 않았는지 어떻게 확인할 수 있나요?"*

**힌트:**
- `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings_stream`을 사용하세요
- 작업 4.2의 셀을 다시 실행하고 다시 세어 보세요 — 결과는 여전히 15여야 합니다

In [ ]:
%sql
-- 전체 행 수 — 15여야 합니다 (5행 × 3개 웨이브 파일)
SELECT COUNT(*) AS total_rows FROM solaris_lab.bronze.solar_readings_stream;

In [ ]:
%sql
-- reading_timestamp 순으로 정렬된 행 샘플 표시
SELECT reading_id, site_id, panel_id, reading_timestamp, power_kw, status
FROM solaris_lab.bronze.solar_readings_stream
ORDER BY reading_timestamp, reading_id;

## ✅ 정리 (선택 사항)

이 랩에서 만든 리소스를 제거하려면 아래 셀을 실행하세요.

> ⚠️ 이 작업은 `solaris_lab` 카탈로그와 모든 테이블 및 볼륨을 포함한 그 안의 모든 콘텐츠를 영구적으로 삭제합니다.

In [ ]:
%sql
-- 모든 랩 리소스를 제거하려면 주석을 해제하고 실행하세요
-- DROP CATALOG IF EXISTS solaris_lab CASCADE;